# Equilibrium shortcut vs. brute-force spin-up — do they agree?

**Question:** if you skipped the fast flux-anchored solver
(`lunar/equilibrium.py`) and instead just ran the old brute-force spin-up for
more and more lunations, would you land on the *same* answer?

**Answer (this notebook demonstrates it): yes.** Both target the one periodic
steady state. The shortcut is an *accelerator*, not a different model.

### What it does
- Computes the **shortcut** profile once — the reference "truth".
- Runs **brute-force** spin-ups from **two initial guesses** (240 K, 260 K)
  over a range of lunation counts, and measures how far each lands from the
  shortcut.
- As lunations grow, **both guesses collapse onto the shortcut** — convergence
  *and* guess-independence in one picture.

### Speed & cores — read before running
A **single** spin-up is *sequential* (lunation *N* needs *N−1*), so it
**cannot** be split across cores. What parallelises here are the many
**independent** runs (2 guesses × several lunation counts). The slowest single
task (the largest `N`) is your wall-time floor no matter how many cores you
have. Set `USE_PARALLEL` / `N_WORKERS` below.

In [ ]:
import sys, os, time, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts" / "figures"))

import make_equilibrium_demo as demo     # the compute library (parallel-safe)
print("project root :", ROOT)
print("CPU cores     :", os.cpu_count())

## 1. Parameters — tune these to trade accuracy for runtime

Smaller `N_GRID` or fewer `GUESSES` = faster. Add `1600` to `N_GRID` to push
panel (b) all the way down to the 0.03 K tolerance line. If multiprocessing
ever misbehaves on your machine, set `USE_PARALLEL = False`.

In [ ]:
N_GRID       = (50, 100, 200, 400, 800)   # lunation counts to test
GUESSES      = (240.0, 260.0)             # two starting guesses -> guess-independence
N_WORKERS    = None                       # None = use all cores; or set an int
USE_PARALLEL = True                        # set False to run serially
KD           = demo.KD                     # representative trial K_d

## 2. Run it

The shortcut runs first (~9 s, serial). Then the brute-force tasks
(`len(GUESSES) * len(N_GRID)` of them) run in parallel across your cores.

In [ ]:
t0 = time.perf_counter()
res = demo.compute_curves(n_grid=N_GRID, guesses=GUESSES,
                          n_workers=N_WORKERS, kd=KD, parallel=USE_PARALLEL)
wall = time.perf_counter() - t0
sc = res["shortcut"]
print(f"shortcut    : {sc['wall']:5.1f} s   (drift {sc['drift']*1e3:.1f} mK, "
      f"closure {sc['closure']:.2%}, {sc['n_outer']} outer iters)")
print(f"brute force : {res['wall_parallel']:5.1f} s wall on {res['n_workers']} core(s) "
      f"for {res['serial_lun']} lunations total")
print(f"total cell  : {wall:5.1f} s")

## 3. The result

In [ ]:
fig = demo.plot_curves(res)
fig

## 4. How to read this

- **Panel (a):** the two curves start far apart (240 K vs 260 K) and both
  settle onto the dashed line — the shortcut's value. After a few hundred
  lunations the starting guess no longer matters.
- **Panel (b):** the worst-case difference (below 0.3 m) between the
  brute-force profile and the shortcut, falling toward the 0.03 K tolerance
  the shortcut is certified to.

**Takeaway:** same destination, ~10× less work — and the shortcut hands you a
convergence certificate (`drift`, `closure`) instead of making you guess how
many lunations is "enough" (the trap that produced the original biased result).

In [ ]:
import numpy as np
print(f"{'guess':>6} {'lunations':>10} {'T@1m [K]':>10} {'|dT|@1m':>9} "
      f"{'max|dT|sub':>11} {'wall[s]':>8}")
for gk, rec in res["curves"].items():
    for n, Tp, dp, ds, w in rec:
        print(f"{gk:6.0f} {n:10.0f} {Tp:10.3f} {dp:9.4f} {ds:11.4f} {w:8.1f}")
print(f"\nshortcut reaches this in {res['shortcut']['wall']:.1f} s.")

## 5. Speeding up the *real* retrieval

The same idea scales to the actual $K_d$ retrieval, where it matters far more:
**every trial $K_d$ in the sweep** (and every sensitivity run) is an
*independent* `run_with()` call, so they can be fanned across cores exactly
like the tasks here. The single forward solve still can't be parallelised — but
you rarely need that: you need *many* solves, and those are embarrassingly
parallel.